# Ranking

In this notebook, we will learn about the features of **PyBroker** that enable you to rank ticker symbols in your trading strategy. With these features, you can easily optimize your strategy and manage risk more effectively.

In [1]:
import pybroker
from pybroker import Strategy, StrategyConfig, YFinance

pybroker.enable_data_source_cache("ranking")

## Scoring Ticker Symbols

In this section, we will learn about how to rank ticker symbols when placing buy orders. Let's begin with an example of how to rank ticker symbols based on volume when placing buy orders. 

In [2]:
def buy_highest_volume(ctx):
    # If there are no long positions across all tickers being traded:
    if not tuple(ctx.long_positions()):
        ctx.buy_shares = ctx.calc_target_shares(1)
        ctx.hold_bars = 2
        ctx.long_score = ctx.volume[-1]

The ```buy_highest_volume``` function ranks ticker symbols by their most recent trading volume and allocates 100% of the portfolio for 2 bars. The ```ctx.score``` is set to ```ctx.volume[-1]```, which is the most recent trading volume.

In [3]:
config = StrategyConfig(max_long_positions=1)
strategy = Strategy(YFinance(), "6/1/2021", "6/1/2022", config)
strategy.add_execution(buy_highest_volume, ["T", "F", "GM", "PFE"])

To limit the number of long positions that can be held at any time to ```1```, we set [max_long_positions](https://www.pybroker.com/en/latest/reference/pybroker.config.html#pybroker.config.StrategyConfig.max_long_positions) to ```1``` in the [StrategyConfig](https://www.pybroker.com/en/latest/reference/pybroker.config.html#pybroker.config.StrategyConfig). In this example, we add the ```buy_highest_volume``` function to the [Strategy](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy) object and specify the ticker symbols to trade: ```['T', 'F', 'GM', 'PFE']```.

In [4]:
result = strategy.backtest()
result.trades

Backtesting: 2021-06-01 00:00:00 to 2022-06-01 00:00:00

Loading bar data...


[*********************100%***********************]  4 of 4 completed


Loaded bar data: 0:00:00 

Test split: 2021-06-01 00:00:00 to 2022-05-31 00:00:00


100% (253 of 253) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Finished backtest: 0:00:01


,type,symbol,entry_date,exit_date,entry,exit,shares,pnl,return_pct,agg_pnl,bars,pnl_per_bar,stop,mae,mfe
id,,,,,,,,,,,,,,,
1,long,F,2021-06-02,2021-06-04,14.85,16.13,6734,8619.52,8.62,8619.52,2,4309.76,bar,-0.17,1.28
2,long,F,2021-06-07,2021-06-09,15.93,15.51,6801,-2856.42,-2.64,5763.10,2,-1428.21,bar,-0.60,0.27
3,long,F,2021-06-10,2021-06-14,15.43,15.06,6832,-2527.84,-2.40,3235.26,2,-1263.92,bar,-0.37,0.35
4,long,F,2021-06-15,2021-06-17,14.96,14.99,6900,207.00,0.20,3442.26,2,103.50,bar,-0.20,0.33
5,long,F,2021-06-18,2021-06-22,14.61,14.96,7003,2451.05,2.40,5893.31,2,1225.53,bar,-0.17,0.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,long,F,2022-05-10,2022-05-12,13.43,12.47,7258,-6967.68,-7.15,-9482.76,2,-3483.84,bar,-0.96,0.41
81,long,F,2022-05-13,2022-05-17,13.25,13.34,6831,614.79,0.68,-8867.97,2,307.40,bar,-0.38,0.38
82,long,F,2022-05-18,2022-05-20,13.03,12.59,6735,-2963.40,-3.38,-11831.37,2,-1481.70,bar,-0.44,0.33


## Shorting the Lowest Scores

**PyBroker** can also rank short orders using [short_score](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.short_score), where orders are placed for the ticker symbols with the *lowest* values. The following example buys the ticker symbol with the highest 5-day rate of change (ROC) while shorting the ticker symbol with the lowest 5-day ROC:

In [5]:
def long_high_short_low(ctx):
    # Wait for 6 bars of data and skip symbols with an open position:
    if ctx.bars < 6 or ctx.long_pos() or ctx.short_pos():
        return
    # Calculate the 5-day rate of change (ROC):
    roc = (ctx.close[-1] - ctx.close[-6]) / ctx.close[-6]
    if roc > 0 and not tuple(ctx.long_positions()):
        ctx.buy_shares = ctx.calc_target_shares(0.5)
        ctx.hold_bars = 2
        ctx.long_score = roc
    elif roc < 0 and not tuple(ctx.short_positions()):
        ctx.sell_shares = ctx.calc_target_shares(0.5)
        ctx.hold_bars = 2
        ctx.short_score = roc


strategy = Strategy(YFinance(), "1/1/2025", "1/1/2026")
strategy.add_execution(long_high_short_low, ["T", "F", "GM", "PFE"])
strategy.set_max_long_positions(1)
strategy.set_max_short_positions(1)
result = strategy.backtest()
result.trades

Backtesting: 2025-01-01 00:00:00 to 2026-01-01 00:00:00

Loading bar data...


[*********************100%***********************]  4 of 4 completed


Loaded bar data: 0:00:00 

Test split: 2025-01-02 00:00:00 to 2025-12-31 00:00:00


100% (250 of 250) |######################| Elapsed Time: 0:00:00 Time:  0:00:00



Finished backtest: 0:00:00


,type,symbol,entry_date,exit_date,entry,exit,shares,pnl,return_pct,agg_pnl,bars,pnl_per_bar,stop,mae,mfe
id,,,,,,,,,,,,,,,
1,long,PFE,2025-01-13,2025-01-15,26.59,26.43,1871,-299.36,-0.60,-299.36,2,-149.68,bar,-0.32,0.28
2,short,T,2025-01-13,2025-01-15,21.54,21.98,2305,-1014.20,-2.00,-1313.56,2,-507.10,bar,-0.44,0.16
3,long,F,2025-01-16,2025-01-21,9.98,10.34,4939,1778.04,3.61,464.48,2,889.02,bar,-0.09,0.36
4,short,PFE,2025-01-16,2025-01-21,26.26,26.51,1881,-470.25,-0.94,-5.77,2,-235.13,bar,-0.31,0.30
5,long,GM,2025-01-22,2025-01-24,52.94,54.15,927,1121.67,2.29,1115.90,2,560.84,bar,-0.51,1.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,short,PFE,2025-12-18,2025-12-22,25.10,25.26,1650,-264.00,-0.63,-17237.70,2,-132.00,bar,-0.42,0.12
143,long,GM,2025-12-19,2025-12-23,81.89,83.05,508,589.28,1.42,-16648.42,2,294.64,bar,-0.80,1.79
144,short,PFE,2025-12-23,2025-12-26,25.09,25.02,1652,115.64,0.28,-16532.78,2,57.82,bar,-0.25,0.26
